# Paper Classification on Colab T4 GPU

End-to-end pipeline: clone repo → install deps → upload data → train 3 tasks → evaluate.

**Required runtime:** GPU T4 (Runtime → Change runtime type → T4 GPU).

**Time:** ~30–45 min for all 3 tasks at 10 epochs each.

**Cost (free tier OK):** ~15 GPU-hours/week limit; full run uses ~0.5h. LLM augmentation already in repo (no API call needed unless re-running).

## 1. Verify GPU

In [ ]:
!nvidia-smi

If you see `Tesla T4` (or `V100`/`A100`) → good. If `command not found` → switch runtime to GPU.

## 2. Clone the repo

In [ ]:
import os
if not os.path.exists('paper-classification'):
    !git clone https://github.com/harnetlinh/paper-classification.git
%cd paper-classification
!ls

## 3. Install dependencies

Colab already has torch + transformers + pandas + scikit-learn. Install the rest.

In [ ]:
!pip install -q pyarrow openpyxl python-dotenv tenacity

## 4. Upload the input Excel

Upload `2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx` from your machine. (~8 MB)

In [ ]:
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'):
    print('Pick the Excel file from your machine:')
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, f'data/{filename}')
!ls -la data/

## 5. Phase 0 — Sanitize

Reads the Excel, builds `gold_2013_2023.parquet` and `main_2024_clean.parquet`. Recovers 417 missing `Total_ID` rows via title lookup.

In [ ]:
!python sanitize.py

## 6. Phase 1 — LLM augmentation (optional, skip if augment parquets shipped in repo)

The repo already ships `outputs/{special_edu,ece,tvet,lll}_augmented.parquet` from a previous run. **Skip this section unless you want to re-augment.**

If you DO want to re-run, set `OPENAI_API_KEY` and uncomment:

In [ ]:
# import os; os.environ['OPENAI_API_KEY'] = 'sk-...'
# !python llm_augment.py --class all

## 7. Phase 2 — Train all 3 models

Hyperparameters (already in `config.py`): LR=2e-5, EPOCHS=10, BATCH=32, AMP fp16, dropout=0.2, best-model selection by macro-AUC, robust threshold tuning (Youden's J fallback for low-support classes).

Time estimate on T4: fields ~12 min, levels ~10 min, method ~10 min.

In [ ]:
!python train_specter2.py --task all

## 8. Phase 3 — Evaluate (val 2023 + test 2024)

Generates `outputs/eval_report.json` with macro F1 / AUC / AP, per-class table, drift gap.

In [ ]:
!python evaluate.py --task all

## 9. Inspect summary

In [ ]:
import json
with open('outputs/eval_report.json') as f:
    rep = json.load(f)
for task, r in rep['tasks'].items():
    print(f"\n=== {task.upper()} ===")
    print(f"  Val 2023:  F1={r['val_2023']['macro_f1']:.4f}  AUC={r['val_2023']['macro_auc']:.4f}  AP={r['val_2023']['macro_ap']:.4f}")
    if 'test_2024' in r:
        print(f"  Test 2024: F1={r['test_2024']['macro_f1']:.4f}  AUC={r['test_2024']['macro_auc']:.4f}  AP={r['test_2024']['macro_ap']:.4f}")

## 10. Download outputs

Save the trained models + reports to your Drive or local machine.

In [ ]:
# Option A: Download as zip
!zip -r outputs.zip outputs/*.json outputs/*.pt outputs/*.parquet 2>/dev/null
from google.colab import files
files.download('outputs.zip')

In [ ]:
# Option B: Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs /content/drive/MyDrive/paper-classification-outputs

## 11. (Optional) Inference on a new Scopus file

Predict labels for a new Excel file with `Title` and `Abstract` columns.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick your new Scopus xlsx
# fname = list(uploaded.keys())[0]
# !python inference.py --input {fname} --output predictions.xlsx
# files.download('predictions.xlsx')